# LLM Pipeline Test — Step 7: Orchestrated Optimization

This notebook uses the **Orchestrator** module to run the full iterative
LLM optimization loop. All logic is encapsulated in:

- `orchestrator/runner.py` — `OptimizationRunner` (main loop)
- `orchestrator/evaluator.py` — `Evaluator` (multi-eval, baselines, mastery)
- `orchestrator/config.json` — all configuration (game, MCTS, optimization)

The notebook simply: **configure → run → evaluate → visualize**.

## Cell 1: Setup — add Tool_Creation to sys.path & import orchestrator

In [1]:
import sys
from pathlib import Path

# Ensure Tool_Creation is on sys.path (parent of orchestrator/)
ROOT = Path(".").resolve().parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from orchestrator import OptimizationRunner, Evaluator
from mcts import MCTSEngine
from mcts.games import Sokoban

print(f"Working dir: {Path('.').resolve()}")
print(f"Tool_Creation root: {ROOT}")
print("All imports OK ✓")

Working dir: /Users/zhuyuezx/Documents/UCSD/Winter_2025/CSE291A/CSE291A_Project/Tool_Creation/orchestrator
Tool_Creation root: /Users/zhuyuezx/Documents/UCSD/Winter_2025/CSE291A/CSE291A_Project/Tool_Creation
All imports OK ✓


## Cell 2: Play one baseline game (sanity check)

Quick check that the engine works before launching the optimization loop.

In [2]:
import json

# Load config for a quick baseline game
with open("config.json") as f:
    cfg = json.load(f)

LEVEL = cfg["game"]["start_level"]
MAX_STEPS = cfg["game"]["constructor_kwargs"]["max_steps"]
ITERS = cfg["mcts"]["iterations"]
MAX_DEPTH = cfg["mcts"]["max_rollout_depth"]
PHASE = cfg["mcts"]["phase"]

game = Sokoban(LEVEL, max_steps=MAX_STEPS)
engine = MCTSEngine(game, iterations=ITERS, max_rollout_depth=MAX_DEPTH, logging=True)
baseline = engine.play_game()

base_tag = "SOLVED" if baseline.get("solved") else "UNSOLVED"
print(f"Baseline ({LEVEL}): {base_tag} in {baseline.get('steps', '?')} steps  "
      f"returns={baseline.get('returns', '?')}")
print(f"Trace: {baseline.get('log_file', 'N/A')}")
print(f"Tools: {list(engine.get_tool_source().keys())}")

Baseline (level3): UNSOLVED in 200 steps  returns=[0.0]
Trace: /Users/zhuyuezx/Documents/UCSD/Winter_2025/CSE291A/CSE291A_Project/Tool_Creation/mcts/records/Sokoban (level3)_20260306_205243_878797.json
Tools: ['selection', 'expansion', 'simulation', 'backpropagation']


## Cell 3: Run iterative optimization

Create the `OptimizationRunner` from `config.json` and run the full loop.
All per-level baselines, mastery tracking, accept/reject, and 70/30 prompting
are handled inside the runner.

In [3]:
runner = OptimizationRunner.from_config("config.json", verbose=True)
summary = runner.run()

best_fn = summary["best_fn"]
print(f"\nbest_fn is {'set' if best_fn else 'None (no improvement adopted)'}")

Starting level: level3
  Computing baseline for level3…
    level3: composite=1.0000, solve_rate=100%, avg_returns=1.0000 (5.5s)
  ⏳ level3 hit 100% on 3 runs — confirming with 7 more games…
  ❌ Confirmation failed: 86% on 7 runs (9.4s)
  Reject floor for level3: 0.5000
  Active levels: ['level1', 'level2', 'level3', 'level4', 'level5', 'level6', 'level7', 'level8', 'level9', 'level10']

############################################################
  ITERATION 1/5, LEVEL=level3
  Baseline composite=1.0000, reject_floor=0.5000
  Active levels: 10/10, mastered: []
############################################################
  Play: UNSOLVED in 200 steps  returns=0.0000  (2.0s)
Step 1/6: Building analysis prompt…
Step 2/6: Querying LLM (step 2 — draft code)…
Step 3/6: Querying LLM (step 3 — critique & finalize)…
  LLM query: status=success  elapsed=37.12s
  Step-1 analysis length: 5545 chars
Step 5/6: Parsing & validating response…
  Optimize: 37.1s
  Eval:  SKIPPED (smoke test failed or e

## Cell 4: Multi-Level Evaluation

Run both **baseline** (default MCTS) and **best LLM-optimized** function
across all levels to measure generalization.

In [5]:
best_fn = summary["best_fn"]
levels  = summary["active_levels"] + list(summary.get("mastered_levels", []))
ev      = runner.evaluator

# ── Evaluate baseline vs optimized on every level ──
rows = []
for lvl in sorted(levels):
    avg_b, sr_b, steps_b, _, t_b = ev.multi_eval(None,     lvl, n=3, logging=False)
    avg_o, sr_o, steps_o, _, t_o = ev.multi_eval(best_fn,  lvl, n=3, logging=False)
    rows.append((lvl, sr_b, sr_o, avg_b, avg_o, steps_b, steps_o, t_b, t_o))
    print(f"{lvl}: baseline={avg_b:.3f} ({sr_b:.0%})  optimized={avg_o:.3f} ({sr_o:.0%})  "
          f"[{t_b:.1f}s + {t_o:.1f}s]")

# ── Summary table ──
print(f"\n{'Level':<10} {'Base Solve%':>12} {'Opt Solve%':>12} "
      f"{'Base AvgRet':>12} {'Opt AvgRet':>12} "
      f"{'Base Steps':>11} {'Opt Steps':>11}")
print("─" * 82)
for lvl, sr_b, sr_o, avg_b, avg_o, steps_b, steps_o, *_ in rows:
    print(f"{lvl:<10} {sr_b*100:>11.0f}% {sr_o*100:>11.0f}% "
          f"{avg_b:>12.3f} {avg_o:>12.3f} "
          f"{steps_b:>11.1f} {steps_o:>11.1f}")

level1: baseline=1.000 (100%)  optimized=1.000 (100%)  [0.0s + 0.0s]
level10: baseline=0.000 (0%)  optimized=0.000 (0%)  [14.5s + 14.5s]
level2: baseline=1.000 (100%)  optimized=1.000 (100%)  [0.0s + 0.0s]
level3: baseline=0.667 (67%)  optimized=0.667 (67%)  [6.0s + 4.5s]
level4: baseline=1.000 (100%)  optimized=1.000 (100%)  [0.3s + 0.2s]
level4: baseline=0.667 (67%)  optimized=1.000 (100%)  [1.0s + 0.2s]
level5: baseline=0.333 (33%)  optimized=0.667 (67%)  [7.4s + 5.3s]
level6: baseline=0.000 (0%)  optimized=0.000 (0%)  [3.8s + 3.8s]
level7: baseline=0.000 (0%)  optimized=0.000 (0%)  [7.6s + 7.6s]
level8: baseline=0.000 (0%)  optimized=0.000 (0%)  [6.7s + 6.7s]
level9: baseline=0.000 (0%)  optimized=0.000 (0%)  [8.3s + 8.0s]

Level       Base Solve%   Opt Solve%  Base AvgRet   Opt AvgRet  Base Steps   Opt Steps
──────────────────────────────────────────────────────────────────────────────────
level1             100%         100%        1.000        1.000         1.0         1.0
level